In [2]:
# Importando as bibliotecas
import requests
import pandas as pd
import numpy as np
import re
import os


# * Listando os parâmetros que serão requisitados da API
parametros = {
    '@trimestre': "'20201'",
    '$top': 511,
    '$format': 'json',
    '$select': 'trimestre,bandeira,funcaoCartao,qtdEstabCredenciados,qtdEstabAtivos'
}

site = 'https://olinda.bcb.gov.br/olinda/servico/MPV_DadosAbertos/versao/v1/odata/EstabCredTransDA(trimestre=@trimestre)'


# * Requisição + Tratamento de Erro 
try:
    response = requests.get(url=site, params=parametros)
    response.raise_for_status()
    data = response.json()
    
    # * Salvando os dados em um DF
    dados_brutos = data['value']
    df = pd.DataFrame(dados_brutos)
    
    # * Tratamento de Dados
    df_copia = df.copy()
    
    
    # Função que insere sublinhado antes de maiúsculas e converte para minúsculas
    def camel_to_snake(name):
        
        # Adiciona '_' antes de maiúsculas e remove espaços extras
        s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
        return re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1).lower()

    # Aplicando a conversão em todas as colunas
    df_copia.columns = [camel_to_snake(col) for col in df.columns]
    
    
    # / Separando o trimestre e o ano em colunas diferentes 
    df_copia['ano'] = df_copia['trimestre'].astype(str).str[-1]
    df_copia['trimestre'] = df_copia['trimestre'].astype(str).str[:4]
    
    # / Alterando o tipo do ano -> int
    df_copia['trimestre'] = df_copia['trimestre'].astype(int)
    df_copia['ano'] = df_copia['ano'].astype(int)
    
    # / Renomeando as colunas
    df_copia = df_copia.rename(columns={'trimestre': 'ano', 'ano': 'trimestre', 'funcao_cartao': 'funcao'})
    
    # / Reordenando as colunas
    coluna_trimestre = df_copia.pop('trimestre')
    df_copia.insert(1, 'trimestre', coluna_trimestre)
    
    
    # * 1. Mapeia qual é o mês e o dia final de cada número de trimestre
    fim_trimestre = {1: '-03-31', 2: '-06-30', 3: '-09-30', 4: '-12-31'}
    
    # * 2. Junta o ano com o sufixo correspondente do trimestre
    df_copia['data_trimestre'] = (
        df_copia['ano'].astype(str) + df_copia['trimestre'].map(fim_trimestre)
    )
    # * 3. Converte para data
    df_copia['data_trimestre'] = pd.to_datetime(df_copia['data_trimestre']).dt.normalize()
    
    
    # / Reordenando coluna data_trimestre
    coluna_data_trimestre = df_copia.pop('data_trimestre')
    df_copia.pop('ano')
    df_copia.insert(0, 'data_trimestre', coluna_data_trimestre)
    
    # * Padronizando os dados da coluna Bandeira
    df_copia['bandeira'] = df_copia['bandeira'].replace({'Bandeira própria': 'Bandeira Própria', 'Visa': 'VISA'})


    display(df_copia['bandeira'].unique())
    display(df_copia.info())
    display(df_copia)

    # Salvando os dados em um arquivo csv
    caminho_csv = os.path.join('..', 'data', 'stg_estabelecimentos.csv')
    df_copia.to_csv(caminho_csv, index=False, sep=';', encoding='utf-8-sig')
    
    print(f'Arquivo salvo com sucesso em: {os.path.abspath(caminho_csv)}')
    
    
except requests.exceptions.RequestException as erro:
    print(f'Erro ao acessar a API: {erro}')

array(['Bandeira Própria', 'Hipercard', 'Elo', 'MasterCard', 'VISA',
       'American Express', 'Outras', 'Diners Club'], dtype=object)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 511 entries, 0 to 510
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   data_trimestre          511 non-null    datetime64[ns]
 1   trimestre               511 non-null    int64         
 2   bandeira                511 non-null    object        
 3   funcao                  511 non-null    object        
 4   qtd_estab_credenciados  511 non-null    int64         
 5   qtd_estab_ativos        511 non-null    int64         
dtypes: datetime64[ns](1), int64(3), object(2)
memory usage: 24.1+ KB


None

,data_trimestre,trimestre,bandeira,funcao,qtd_estab_credenciados,qtd_estab_ativos
0,2020-03-31,1,Bandeira Própria,Pré-pago,323089,127013
1,2020-03-31,1,Bandeira Própria,Débito,388644,151161
2,2020-03-31,1,Hipercard,Débito,1687487,1568
3,2020-03-31,1,Elo,Pré-pago,8199488,2186124
4,2020-03-31,1,MasterCard,Crédito,24300602,7281475
...,...,...,...,...,...,...
506,2026-03-31,1,Outras,Crédito,27119732,720774
507,2026-03-31,1,Bandeira Própria,Crédito,2036402,296986
508,2026-03-31,1,VISA,Pré-pago,78177611,6710276
509,2026-03-31,1,MasterCard,Crédito,107035544,15271464


Arquivo salvo com sucesso em: c:\Users\mathe\OneDrive\Documentos\Meus Projetos\Análise de Dados\Projeto end-to-end\data\stg_estabelecimentos.csv
